In [ ]:
!pip install -q open_clip_torch torch torchvision

import numpy as np
import torch
import torch.nn.functional as F
from google.colab import drive
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import accuracy_score, classification_report

drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/diss-datasets"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

dinov2 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
dinov2 = dinov2.to(device).eval()
print("DINOv2 ViT-B/14 loaded")

def series_to_image(x, patch_len=None, img_size=224):
    N, T = x.shape
    P = patch_len or int(np.sqrt(T))
    M = T // P
    x = x[:, :M * P].reshape(N, M, P)
    img = torch.tensor(x, dtype=torch.float32).unsqueeze(1)   # (N, 1, M, P)
    img = (img - img.mean()) / (img.std() + 1e-6)
    img = img.repeat(1, 3, 1, 1)                                # (N, 3, M, P)
    img = F.interpolate(img, size=(img_size, img_size),
                         mode='bilinear', align_corners=False)
    return img

@torch.no_grad()
def extract_dinov2(X, batch_size=32):
    N, C, T = X.shape
    embs = []
    for i in range(0, N, batch_size):
        chunk = X[i:i+batch_size]
        per_channel = []
        for c in range(C):
            img = series_to_image(chunk[:, c, :]).to(device)
            feat = dinov2(img)
            per_channel.append(feat.cpu().numpy())
        embs.append(np.concatenate(per_channel, axis=1))
    return np.concatenate(embs, axis=0)

In [ ]:
data_cmj = np.load(f'{SAVE_DIR}/CMJ.npy', allow_pickle=True).item()
X_tr, y_tr = data_cmj['train']['X'], np.array(data_cmj['train']['y'])
X_te, y_te = data_cmj['test']['X'], np.array(data_cmj['test']['y'])

Z_tr = extract_dinov2(X_tr)
Z_te = extract_dinov2(X_te)

clf = RidgeClassifier()
clf.fit(Z_tr, y_tr)
y_pred = clf.predict(Z_te)

acc = accuracy_score(y_te, y_pred)
print(f'TiViT-DINOv2 Accuracy on CMJ: {acc:.4f}')
print(classification_report(y_te, y_pred))

np.save(f'{SAVE_DIR}/TiViT_DINOv2_CMJ_train_emb.npy', Z_tr)
np.save(f'{SAVE_DIR}/TiViT_DINOv2_CMJ_test_emb.npy', Z_te)

In [ ]:
baseline_datasets = ['MP8', 'MP50', 'SYNTH', 'Rowing']
dinov2_results = {'CMJ': 0.9385}

for name in baseline_datasets:
    print(f"\n=== {name} ===")
    d = np.load(f'{SAVE_DIR}/{name}.npy', allow_pickle=True).item()
    Xtr, ytr = d['train']['X'], np.array(d['train']['y'])
    Xte, yte = d['test']['X'], np.array(d['test']['y'])

    Ztr = extract_dinov2(Xtr)
    Zte = extract_dinov2(Xte)

    clf = RidgeClassifier()
    clf.fit(Ztr, ytr)
    acc = accuracy_score(yte, clf.predict(Zte))
    dinov2_results[name] = acc
    print(f'TiViT-DINOv2 Accuracy on {name}: {acc:.4f}')

    np.save(f'{SAVE_DIR}/TiViT_DINOv2_{name}_train_emb.npy', Ztr)
    np.save(f'{SAVE_DIR}/TiViT_DINOv2_{name}_test_emb.npy', Zte)

print("\n=== CHB-MIT v2 (this is the long one, 18 channels) ===")
import time
t0 = time.time()

data_chb = np.load(f'{SAVE_DIR}/CHBMIT_crosspatient_v2.npy', allow_pickle=True).item()
Xtr_c = data_chb['train']['X']
ytr_c = np.array([int(v) for v in data_chb['train']['y']])
Xte_c = data_chb['test']['X']
yte_c = np.array([int(v) for v in data_chb['test']['y']])

print(f"train windows: {Xtr_c.shape[0]}, test windows: {Xte_c.shape[0]}, channels: {Xtr_c.shape[1]}")
print("extracting train set...")
Ztr_c = extract_dinov2(Xtr_c)
print(f"train done in {(time.time()-t0)/60:.1f} min, extracting test set...")
Zte_c = extract_dinov2(Xte_c)
print(f"both done in {(time.time()-t0)/60:.1f} min total")

clf_c = RidgeClassifier()
clf_c.fit(Ztr_c, ytr_c)
acc_c = accuracy_score(yte_c, clf_c.predict(Zte_c))
dinov2_results['CHB-MIT'] = acc_c
print(f'TiViT-DINOv2 Accuracy on CHB-MIT v2: {acc_c:.4f}')
print(classification_report(yte_c, clf_c.predict(Zte_c)))

np.save(f'{SAVE_DIR}/TiViT_DINOv2_CHBMIT_v2_train_emb.npy', Ztr_c)
np.save(f'{SAVE_DIR}/TiViT_DINOv2_CHBMIT_v2_test_emb.npy', Zte_c)

print("\nAll DINOv2 results:", dinov2_results)

In [ ]:
data_chb = np.load(f'{SAVE_DIR}/CHBMIT_crosspatient_v2.npy', allow_pickle=True).item()
te = data_chb['test']
y_te_chb = te['y'].astype(int)
file = np.asarray(te['file'])
t0 = te['t0'].astype(int)

MERGE = 30
event_id = np.full(len(y_te_chb), -1, dtype=int)
next_ev = 0
for f in np.unique(file):
    idx = np.where((file == f) & (y_te_chb == 1))[0]
    if len(idx) == 0:
        continue
    order = idx[np.argsort(t0[idx])]
    starts = t0[order]
    ev_local = np.zeros(len(order), dtype=int)
    for i in range(1, len(order)):
        ev_local[i] = ev_local[i-1] + (1 if (starts[i] - starts[i-1]) > MERGE else 0)
    for local, gi in zip(ev_local, order):
        event_id[gi] = next_ev + local
    next_ev += int(ev_local.max()) + 1

n_events = next_ev
neg_mask = (y_te_chb == 0)

def sens_at_fpr(y_prob, fpr_targets=(0.05, 0.10, 0.20)):
    rows = []
    for thr in np.unique(y_prob):
        pred = y_prob >= thr
        det = np.array([pred[event_id == ev].any() for ev in range(n_events)])
        rows.append((thr, det.mean(), pred[neg_mask].mean()))
    rows = np.array(rows)
    out = {}
    for tgt in fpr_targets:
        ok = rows[rows[:, 2] <= tgt]
        out[tgt] = ok[:, 1].max() if len(ok) else np.nan
    return out, rows

prob_dinov2 = clf_c.decision_function(Zte_c)
res_dinov2, _ = sens_at_fpr(prob_dinov2)

print("TiViT-DINOv2 event-level sensitivity:")
for tgt, s in res_dinov2.items():
    print(f"  at {int(tgt*100)}% interictal FPR: sensitivity = {s:.3f}  ({int(round(s*n_events))}/{n_events} events)")

In [ ]:
from openpyxl import load_workbook

XLSX = "/content/drive/MyDrive/results.xlsx"
wb = load_workbook(XLSX)
ws = wb["RQ3 event sensitivity"]

next_row = ws.max_row + 1
row = ("TiViT-DINOv2", "channel-independent (image, per-channel)", res_dinov2[0.05], res_dinov2[0.10], res_dinov2[0.20])
for c, v in enumerate(row, 1):
    cell = ws.cell(row=next_row, column=c, value=v)
    if c > 2:
        cell.number_format = "0.000"

wb.save(XLSX)
print("saved, table now has", ws.max_row - 3, "rows")

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import Font

XLSX = "/content/drive/MyDrive/results.xlsx"
wb = load_workbook(XLSX)
bold = Font(bold=True)
sheet_name = "TiViT backbone comparison"
if sheet_name in wb.sheetnames:
    del wb[sheet_name]
ws = wb.create_sheet(sheet_name)

ws.cell(row=1, column=1, value="TiViT: CLIP vs DINOv2 backbone, frozen linear probe accuracy").font = bold
headers = ["Dataset", "TiViT-CLIP", "TiViT-DINOv2", "Difference"]
for c, h in enumerate(headers, 1):
    ws.cell(row=3, column=c, value=h).font = bold

clip_results = {"CMJ": 0.9330, "MP8": 0.5613, "MP50": 0.6672, "SYNTH": 0.8470, "Rowing": 0.7697}
r = 4
for ds in ["CMJ", "MP8", "MP50", "SYNTH", "Rowing", "CHB-MIT"]:
    clip_v = clip_results.get(ds)
    dino_v = dinov2_results[ds]
    diff = (dino_v - clip_v) if clip_v is not None else None
    for c, v in enumerate((ds, clip_v, dino_v, diff), 1):
        cell = ws.cell(row=r, column=c, value=v)
        if c > 1 and v is not None:
            cell.number_format = "0.000"
    r += 1

wb.save(XLSX)
print("saved")